# Notebook-first application walkthrough

**Problem / objective:** Ingest events reliably despite duplicates, invalid records, late arrivals and replayed batches.

**Decision / solution:** Accept valid events once, quarantine bad records, reconcile counts and make replay behaviour observable.

This front section is intentionally analysis-first. It uses direct notebook code for inspection, EDA, visualisation and evidence review. The original notebook work is preserved below, followed by modular production code where that adds engineering evidence.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
PROJECT_SLUG = 'reliable_event_pipeline'
ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    candidate = ROOT.parent.parent if ROOT.name == PROJECT_SLUG else ROOT
    if (candidate / 'projects').exists():
        ROOT = candidate
PROJECT = ROOT / 'projects' / PROJECT_SLUG
if not PROJECT.exists() and Path.cwd().name == PROJECT_SLUG:
    PROJECT = Path.cwd()
    ROOT = PROJECT.parent.parent
assert PROJECT.exists(), f'Project directory not found: {PROJECT}'
print('Repository root:', ROOT.resolve())
print('Project:', PROJECT.resolve())


## 1. Find the real data and retained evidence

Instead of hiding the dataset behind a helper function, start by seeing what the project actually ships: raw/small data, fixtures, outputs, results and verified evidence. External large datasets remain reproducibly downloadable from the documented source.


In [ ]:
candidate_files = []
for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
    candidate_files.extend(PROJECT.rglob(pattern))
verified_dir = ROOT / 'verified' / PROJECT_SLUG
if verified_dir.exists():
    for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
        candidate_files.extend(verified_dir.rglob(pattern))
candidate_files = sorted({p.resolve() for p in candidate_files if p.is_file()})
file_inventory = pd.DataFrame({
    'file': [str(p.relative_to(ROOT)) if ROOT in p.parents else str(p) for p in candidate_files],
    'suffix': [p.suffix.lower() for p in candidate_files],
    'size_kb': [round(p.stat().st_size / 1024, 1) for p in candidate_files],
})
display(file_inventory.head(40))
print(f'Inspectable local data/evidence files: {len(file_inventory):,}')


## 2. Direct tabular data audit

The code below deliberately avoids a project-specific wrapper. It opens the first sensible local tabular asset, shows its schema and quality profile, and makes the data issues visible before modelling. If the full raw dataset is external, run the project's documented download cell/entry point first and rerun this section.


In [ ]:
tabular_candidates = [p for p in candidate_files if p.suffix.lower() in {'.csv', '.tsv', '.parquet'}]
preferred = [p for p in tabular_candidates if not any(token in p.name.lower() for token in ('metric', 'summary', 'verification'))]
tabular_path = (preferred or tabular_candidates or [None])[0]
df = None
if tabular_path is not None:
    if tabular_path.suffix.lower() == '.parquet':
        df = pd.read_parquet(tabular_path)
    else:
        sep = '\t' if tabular_path.suffix.lower() == '.tsv' else ','
        df = pd.read_csv(tabular_path, sep=sep, nrows=200_000)
    print('Loaded:', tabular_path)
    print('Shape:', df.shape)
    display(df.head())
    audit = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing': df.isna().sum(),
        'missing_pct': (100 * df.isna().mean()).round(2),
        'unique': df.nunique(dropna=False),
    }).sort_values(['missing_pct', 'unique'], ascending=[False, False])
    display(audit.head(30))
    print('Duplicate rows:', int(df.duplicated().sum()))
else:
    print('No local CSV/TSV/Parquet found yet. Use the project README/run path to download or build the documented dataset, then rerun this audit.')


## 3. Exploratory data analysis and visualisation

These plots are intentionally created in the notebook rather than described in prose. They expose distribution, missingness, scale, category balance and numeric relationships before any final model decision.


In [ ]:
if df is not None and len(df):
    missing_pct = (100 * df.isna().mean()).sort_values(ascending=False).head(20)
    missing_pct = missing_pct[missing_pct > 0]
    if len(missing_pct):
        plt.figure(figsize=(10, 4))
        missing_pct.plot(kind='bar')
        plt.title('Missing values by feature (%)')
        plt.ylabel('Missing %')
        plt.xticks(rotation=60, ha='right')
        plt.tight_layout()
        plt.show()

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:8]
    for col in numeric_cols:
        series = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(series):
            plt.figure(figsize=(8, 4))
            plt.hist(series, bins=30, alpha=0.8)
            plt.axvline(series.median(), linestyle='--', label=f'median={series.median():.2f}')
            plt.title(f'Distribution: {col}')
            plt.xlabel(col)
            plt.ylabel('Count')
            plt.legend()
            plt.tight_layout()
            plt.show()

    categorical_cols = [c for c in df.columns if c not in numeric_cols and df[c].nunique(dropna=False) <= 30][:4]
    for col in categorical_cols:
        counts = df[col].fillna('<missing>').astype(str).value_counts().head(15)
        plt.figure(figsize=(9, 4))
        counts.sort_values().plot(kind='barh')
        plt.title(f'Top categories: {col}')
        plt.xlabel('Rows')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr(numeric_only=True)
        plt.figure(figsize=(8, 6))
        image = plt.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
        plt.colorbar(image, label='Correlation')
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=60, ha='right')
        plt.yticks(range(len(corr.index)), corr.index)
        plt.title('Numeric correlation matrix')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        x_col, y_col = numeric_cols[0], numeric_cols[-1]
        sample = df[[x_col, y_col]].dropna().sample(min(3000, len(df.dropna(subset=[x_col, y_col]))), random_state=42)
        if len(sample):
            plt.figure(figsize=(7, 5))
            plt.scatter(sample[x_col], sample[y_col], alpha=0.35, s=18)
            plt.xlabel(x_col)
            plt.ylabel(y_col)
            plt.title(f'{y_col} versus {x_col}')
            plt.tight_layout()
            plt.show()
else:
    print('Run the documented data-build/download path, then rerun this section to render raw-data EDA.')


## 4. Inspect the measured results, not just the code

A portfolio project is stronger when it retains evidence. This section reads machine-readable JSON/CSV outputs and turns scalar metrics into a quick visual comparison.


In [ ]:
json_files = [p for p in candidate_files if p.suffix.lower() == '.json']
metric_rows = []
for path in json_files[:30]:
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        continue
    stack = [('', payload)]
    while stack:
        prefix, value = stack.pop()
        if isinstance(value, dict):
            for key, child in value.items():
                stack.append((f'{prefix}.{key}' if prefix else str(key), child))
        elif isinstance(value, (int, float)) and not isinstance(value, bool) and np.isfinite(value):
            metric_rows.append({
                'file': str(path.relative_to(ROOT)) if ROOT in path.parents else str(path),
                'metric': prefix,
                'value': float(value),
            })
metrics_df = pd.DataFrame(metric_rows)
if len(metrics_df):
    display(metrics_df.head(40))
    plot_df = metrics_df[np.isfinite(metrics_df['value'])].copy()
    plot_df = plot_df[plot_df['value'].abs() < 1_000_000].head(20)
    if len(plot_df):
        labels = (plot_df['file'].str.split('/').str[-1] + ' :: ' + plot_df['metric']).tolist()
        plt.figure(figsize=(10, max(4, 0.35 * len(plot_df))))
        plt.barh(range(len(plot_df)), plot_df['value'])
        plt.yticks(range(len(plot_df)), labels)
        plt.title('Retained project metrics / evidence')
        plt.tight_layout()
        plt.show()
else:
    print('No scalar JSON evidence found. Run the project and retain metrics/results before treating it as complete.')


## 5. Reproduce the application

The notebook should be understandable without running anything, but a reviewer can reproduce the canonical application below. The switch is off by default so opening the notebook never triggers a long training job unexpectedly.


In [ ]:
RUN_PROJECT = False
entrypoint = PROJECT / 'run.py'
if RUN_PROJECT and entrypoint.exists():
    subprocess.run([sys.executable, str(entrypoint)], cwd=PROJECT, check=True)
elif entrypoint.exists():
    print(f'Reproduce with: cd {PROJECT} && {sys.executable} run.py')
else:
    print('This project uses a different documented entry point; see README.md in the project folder.')


## 6. Decision / solution

Accept valid events once, quarantine bad records, reconcile counts and make replay behaviour observable.

The final recommendation should be tied to the measured validation evidence and error analysis below. A model is not the solution by itself; the solution is the decision process built around it.


# Reliable Event Pipeline — Full Python Code

**Hiring purpose:** one project, one notebook, with the actual Python implementation visible. The modular files remain in the repository because that is how production code should be organised; this notebook mirrors those files so a recruiter can inspect the full code without hunting.


## Dataset and reproducibility

Committed event fixtures in fixtures/batch_1.csv and fixtures/batch_2.csv.

The project README/data card documents provenance, constraints and the exact reproduction path. Large third-party raw files are not duplicated in Git when licensing or repository size makes that poor engineering practice.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

PROJECT_SLUG = 'reliable_event_pipeline'
ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    target = Path('/content/uni_projects')
    if not target.exists():
        subprocess.run(['git', 'clone', 'https://github.com/Jorgoluka100/uni_projects.git', str(target)], check=True)
    os.chdir(target)
    ROOT = target
PROJECT = ROOT / 'projects' / PROJECT_SLUG
assert PROJECT.exists(), PROJECT
print('Project:', PROJECT.resolve())


## Full Python implementation

Every code cell below is copied directly from the corresponding `.py` file on the same commit. These cells are intentionally tagged `source-mirror` so the notebook acts as a readable code portfolio while the canonical modules remain testable files.


### `run.py`


In [ ]:
from __future__ import annotations

import argparse
import json
import tempfile
from pathlib import Path

from src.pipeline import connect, ingest_csv, quality_summary

ROOT = Path(__file__).resolve().parent


def run_demo(db_path: Path) -> dict[str, object]:
    conn = connect(db_path)
    first = ingest_csv(conn, ROOT / "fixtures" / "batch_1.csv", batch_name="batch_1")
    second = ingest_csv(conn, ROOT / "fixtures" / "batch_2.csv", batch_name="batch_2")
    summary = quality_summary(conn)
    conn.close()
    return {
        "batches": [first.__dict__, second.__dict__],
        "quality": summary,
        "verification_pass": bool(summary["verification_pass"]),
    }


def self_test() -> None:
    with tempfile.TemporaryDirectory() as tmp:
        result = run_demo(Path(tmp) / "events.db")
    assert result["verification_pass"] is True
    assert result["quality"]["warehouse_rows"] == 7
    assert result["quality"]["duplicate_event_ids"] == 0
    assert result["quality"]["null_customer_ids"] == 0
    assert result["quality"]["late_rows"] == 1
    assert result["quality"]["rejected_rows"] == 2
    print("self-test passed")


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--db", type=Path, default=ROOT / "artifacts" / "events.db")
    parser.add_argument("--output", type=Path, default=ROOT / "results" / "verified_run.json")
    parser.add_argument("--self-test", action="store_true")
    args = parser.parse_args()

    if args.self_test:
        self_test()
        return 0

    args.db.parent.mkdir(parents=True, exist_ok=True)
    args.output.parent.mkdir(parents=True, exist_ok=True)
    if args.db.exists():
        args.db.unlink()
    result = run_demo(args.db)
    args.output.write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(json.dumps(result, indent=2))
    return 0 if result["verification_pass"] else 1


if __name__ == "__main__":
    raise SystemExit(main())


### `src/pipeline.py`


In [ ]:
from __future__ import annotations

import csv
import json
import sqlite3
from dataclasses import asdict, dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path

REQUIRED_COLUMNS = (
    "event_id",
    "customer_id",
    "event_time",
    "source",
    "event_type",
    "amount",
)


@dataclass(frozen=True)
class BatchMetrics:
    batch_name: str
    input_rows: int
    valid_rows: int
    rejected_rows: int
    duplicate_rows_in_batch: int
    existing_rows_skipped: int
    late_rows: int
    inserted_rows: int
    warehouse_rows_after: int


def connect(db_path: str | Path) -> sqlite3.Connection:
    conn = sqlite3.connect(str(db_path))
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


def initialise(conn: sqlite3.Connection) -> None:
    conn.executescript(
        """
        CREATE TABLE IF NOT EXISTS events (
            event_id TEXT PRIMARY KEY,
            customer_id TEXT NOT NULL,
            event_time TEXT NOT NULL,
            source TEXT NOT NULL,
            event_type TEXT NOT NULL,
            amount REAL NOT NULL,
            is_late INTEGER NOT NULL CHECK (is_late IN (0, 1)),
            ingested_at TEXT NOT NULL,
            batch_name TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS rejected_events (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            batch_name TEXT NOT NULL,
            row_number INTEGER NOT NULL,
            reason TEXT NOT NULL,
            payload TEXT NOT NULL,
            rejected_at TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS batch_audit (
            batch_name TEXT PRIMARY KEY,
            input_rows INTEGER NOT NULL,
            valid_rows INTEGER NOT NULL,
            rejected_rows INTEGER NOT NULL,
            duplicate_rows_in_batch INTEGER NOT NULL,
            existing_rows_skipped INTEGER NOT NULL,
            late_rows INTEGER NOT NULL,
            inserted_rows INTEGER NOT NULL,
            warehouse_rows_after INTEGER NOT NULL,
            completed_at TEXT NOT NULL
        );
        """
    )
    conn.commit()


def _parse_utc(value: str) -> datetime:
    text = value.strip().replace("Z", "+00:00")
    dt = datetime.fromisoformat(text)
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)


def _current_max_event_time(conn: sqlite3.Connection) -> datetime | None:
    row = conn.execute("SELECT MAX(event_time) AS max_event_time FROM events").fetchone()
    if not row or row["max_event_time"] is None:
        return None
    return _parse_utc(row["max_event_time"])


def _normalise(row: dict[str, str]) -> dict[str, object]:
    missing_columns = [name for name in REQUIRED_COLUMNS if name not in row]
    if missing_columns:
        raise ValueError(f"missing columns: {', '.join(missing_columns)}")

    event_id = row["event_id"].strip()
    customer_id = row["customer_id"].strip()
    source = row["source"].strip().lower()
    event_type = row["event_type"].strip().lower()

    if not event_id:
        raise ValueError("missing event_id")
    if not customer_id:
        raise ValueError("missing customer_id")
    if not source:
        raise ValueError("missing source")
    if not event_type:
        raise ValueError("missing event_type")

    event_time = _parse_utc(row["event_time"])
    amount = float(row["amount"])
    if amount < 0:
        raise ValueError("negative amount")

    return {
        "event_id": event_id,
        "customer_id": customer_id,
        "event_time": event_time,
        "source": source,
        "event_type": event_type,
        "amount": amount,
    }


def ingest_csv(
    conn: sqlite3.Connection,
    csv_path: str | Path,
    *,
    batch_name: str,
    allowed_lateness_hours: int = 48,
) -> BatchMetrics:
    """Validate one CSV batch and load it idempotently into the warehouse.

    Late arrivals are accepted but flagged. Rows older than the current warehouse
    maximum event time minus ``allowed_lateness_hours`` are considered late.
    """
    initialise(conn)
    if conn.execute("SELECT 1 FROM batch_audit WHERE batch_name = ?", (batch_name,)).fetchone():
        raise ValueError(f"batch_name already processed: {batch_name}")

    previous_max = _current_max_event_time(conn)
    watermark = previous_max - timedelta(hours=allowed_lateness_hours) if previous_max else None
    now = datetime.now(timezone.utc).isoformat()

    input_rows = valid_rows = rejected_rows = 0
    duplicate_rows_in_batch = existing_rows_skipped = late_rows = inserted_rows = 0
    seen_ids: set[str] = set()

    with Path(csv_path).open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        missing_headers = [name for name in REQUIRED_COLUMNS if name not in (reader.fieldnames or [])]
        if missing_headers:
            raise ValueError(f"missing required headers: {', '.join(missing_headers)}")

        for row_number, raw in enumerate(reader, start=2):
            input_rows += 1
            try:
                clean = _normalise(raw)
            except Exception as exc:
                rejected_rows += 1
                conn.execute(
                    "INSERT INTO rejected_events(batch_name,row_number,reason,payload,rejected_at) VALUES(?,?,?,?,?)",
                    (batch_name, row_number, str(exc), json.dumps(raw, sort_keys=True), now),
                )
                continue

            event_id = str(clean["event_id"])
            if event_id in seen_ids:
                duplicate_rows_in_batch += 1
                rejected_rows += 1
                conn.execute(
                    "INSERT INTO rejected_events(batch_name,row_number,reason,payload,rejected_at) VALUES(?,?,?,?,?)",
                    (batch_name, row_number, "duplicate event_id within batch", json.dumps(raw, sort_keys=True), now),
                )
                continue
            seen_ids.add(event_id)
            valid_rows += 1

            event_dt = clean["event_time"]
            assert isinstance(event_dt, datetime)
            is_late = int(watermark is not None and event_dt < watermark)
            late_rows += is_late

            before = conn.total_changes
            conn.execute(
                """
                INSERT OR IGNORE INTO events(
                    event_id, customer_id, event_time, source, event_type,
                    amount, is_late, ingested_at, batch_name
                ) VALUES(?,?,?,?,?,?,?,?,?)
                """,
                (
                    clean["event_id"],
                    clean["customer_id"],
                    event_dt.isoformat(),
                    clean["source"],
                    clean["event_type"],
                    clean["amount"],
                    is_late,
                    now,
                    batch_name,
                ),
            )
            if conn.total_changes > before:
                inserted_rows += 1
            else:
                existing_rows_skipped += 1

    warehouse_rows_after = conn.execute("SELECT COUNT(*) FROM events").fetchone()[0]
    metrics = BatchMetrics(
        batch_name=batch_name,
        input_rows=input_rows,
        valid_rows=valid_rows,
        rejected_rows=rejected_rows,
        duplicate_rows_in_batch=duplicate_rows_in_batch,
        existing_rows_skipped=existing_rows_skipped,
        late_rows=late_rows,
        inserted_rows=inserted_rows,
        warehouse_rows_after=warehouse_rows_after,
    )
    conn.execute(
        """
        INSERT INTO batch_audit(
            batch_name,input_rows,valid_rows,rejected_rows,duplicate_rows_in_batch,
            existing_rows_skipped,late_rows,inserted_rows,warehouse_rows_after,completed_at
        ) VALUES(?,?,?,?,?,?,?,?,?,?)
        """,
        (*asdict(metrics).values(), now),
    )
    conn.commit()
    return metrics


def quality_summary(conn: sqlite3.Connection) -> dict[str, object]:
    initialise(conn)
    total = conn.execute("SELECT COUNT(*) FROM events").fetchone()[0]
    duplicate_ids = conn.execute(
        "SELECT COUNT(*) FROM (SELECT event_id FROM events GROUP BY event_id HAVING COUNT(*) > 1)"
    ).fetchone()[0]
    null_business_keys = conn.execute(
        "SELECT COUNT(*) FROM events WHERE customer_id IS NULL OR TRIM(customer_id) = ''"
    ).fetchone()[0]
    late_rows = conn.execute("SELECT COUNT(*) FROM events WHERE is_late = 1").fetchone()[0]
    rejected = conn.execute("SELECT COUNT(*) FROM rejected_events").fetchone()[0]
    revenue = conn.execute("SELECT ROUND(SUM(amount), 2) FROM events").fetchone()[0] or 0.0
    return {
        "warehouse_rows": total,
        "duplicate_event_ids": duplicate_ids,
        "null_customer_ids": null_business_keys,
        "late_rows": late_rows,
        "rejected_rows": rejected,
        "total_amount": revenue,
        "verification_pass": duplicate_ids == 0 and null_business_keys == 0,
    }


## Run the real project

The cell below executes the canonical project entry point rather than a rewritten toy version. Keep `RUN_PIPELINE = False` when you only want to inspect the notebook; change it to `True` to reproduce the project.


In [ ]:
RUN_PIPELINE = False
if RUN_PIPELINE:
    subprocess.run([sys.executable, 'run.py'], cwd=PROJECT, check=True)
else:
    print(f'Reproduce with: cd {PROJECT} && python run.py')


In [ ]:
evidence = []
for folder in (PROJECT / 'results', ROOT / 'verified' / PROJECT_SLUG):
    if folder.exists():
        evidence.extend(sorted(folder.glob('*.json')))
for path in evidence[:5]:
    print('\n---', path.relative_to(ROOT), '---')
    print(path.read_text(encoding='utf-8')[:12000])


## Interview discussion

Be ready to explain the business problem, dataset provenance, cleaning/preprocessing decisions, leakage controls, modelling or analytical choices, evaluation design, limitations, testing strategy and what you would change in production. The key signal is that the notebook, modular source, tests and retained evidence all tell the same story.


# Engineering appendix — canonical application source

The analysis and visual evidence come first. The cells below preserve additional canonical Python from this project for reviewers who want to inspect pipelines, APIs, tests, feature code, monitoring and reusable implementation details.


## Canonical source: `tests/test_pipeline.py`


In [ ]:
from __future__ import annotations

import tempfile
import unittest
from pathlib import Path

from src.pipeline import connect, ingest_csv, quality_summary

ROOT = Path(__file__).resolve().parents[1]


class PipelineTests(unittest.TestCase):
    def test_two_batches_are_clean_and_idempotent_by_event_id(self) -> None:
        with tempfile.TemporaryDirectory() as tmp:
            conn = connect(Path(tmp) / "events.db")
            first = ingest_csv(conn, ROOT / "fixtures" / "batch_1.csv", batch_name="batch_1")
            second = ingest_csv(conn, ROOT / "fixtures" / "batch_2.csv", batch_name="batch_2")
            summary = quality_summary(conn)

        self.assertEqual(first.inserted_rows, 4)
        self.assertEqual(first.rejected_rows, 2)
        self.assertEqual(first.duplicate_rows_in_batch, 1)
        self.assertEqual(second.inserted_rows, 3)
        self.assertEqual(second.existing_rows_skipped, 1)
        self.assertEqual(second.late_rows, 1)
        self.assertEqual(summary["warehouse_rows"], 7)
        self.assertEqual(summary["duplicate_event_ids"], 0)
        self.assertEqual(summary["null_customer_ids"], 0)
        self.assertTrue(summary["verification_pass"])

    def test_reusing_a_batch_name_is_blocked(self) -> None:
        with tempfile.TemporaryDirectory() as tmp:
            conn = connect(Path(tmp) / "events.db")
            ingest_csv(conn, ROOT / "fixtures" / "batch_1.csv", batch_name="batch_1")
            with self.assertRaises(ValueError):
                ingest_csv(conn, ROOT / "fixtures" / "batch_1.csv", batch_name="batch_1")


if __name__ == "__main__":
    unittest.main()


# Portfolio depth check

**Meaningful code lines visible in this notebook:** 488. For a major recruiter-facing application the working target is roughly **1,000 meaningful lines**, with a practical guide of about 600–1,400 depending on the problem. This notebook is below the major-project guide and should grow only through substantive analysis/application depth.

Line count is not a quality metric by itself. Add code only when it improves the real project: data acquisition, validation, cleaning, EDA, visualisation, feature engineering, baselines, model comparison, tuning, leakage control, error analysis, explainability, uncertainty, inference, tests, monitoring, deployment or decision logic.
